## Zadania: Modele Multimodalne

Przykłady przetwarzania obrazu przez model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/exercises/pl/exercise_4_1_Multimodal_models.ipynb)

| | |
|---|---|
| 🎯 **Czego poćwiczysz** | Opis obrazu przez model multimodalny i pytania o zawartość obrazu. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | bardzo niski – kilka krótkich zapytań |
| ⏱️ **Szacowany czas** | ~25 min (z rozwiązaniem zadań) |

Zadanie 1 \
Wczytaj dowolny obraz z dysku (np. `sample.jpg`) i poproś model o opis w 1–2 zdaniach.

In [ ]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU openai python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

# Dane przykładowe są pobierane automatycznie w Colab; lokalnie użyj katalogu głównego repozytorium.
import os
import subprocess
import sys
from pathlib import Path

if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
    repo_root = Path("/content/course_llm_agent_apps_with_langchain_and_langgraph")
    if not (repo_root / "data").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph.git",
            str(repo_root),
        ], check=True, stdout=subprocess.DEVNULL)
else:
    repo_root = next((
        parent for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir()
    ), Path.cwd())
os.chdir(repo_root)

In [6]:
# TODO: Wczytaj obraz i zbuduj wiadomość dla modelu multimodalnego
# HINT: Możesz zaczytać obrazek i przekazać jako data URL lub file-URI (w zależności od klienta).
from base64 import b64encode
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

img_path = Path("data/img/airport.jpg")  # <- podmień na własną ścieżkę
if img_path.exists():
    client = OpenAI()
    data = b64encode(img_path.read_bytes()).decode("utf-8")
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": ("Opisz co jest na obrazku w 1–2 zdaniach po polsku.")},
                {"type": "input_image", "image_url": f"data:image/jpeg;base64,{data}"}
            ],
        }],
    )

    response = resp.output_text
    print("Odpowiedź modelu:", response)
else:
    print("Brak pliku – wstaw własny obraz i uruchom ponownie.")

Model decision: Na obrazku widać ludzi poruszających się w przestrzeni lotniska, przechodzących wzdłuż dużych okien. Silhouetty pasażerów z bagażami kontrastują z jasnym tłem.


Zadanie 2 \
Łączenie modalności – obraz + pytanie \
Podaj pytanie o wybrany fragment obrazka (np. „Ile jest osób?”) i poproś model o krótką odpowiedź.

In [11]:
# TODO: Zastosuj podobny schemat jak w Zadaniu 1, ale dodaj pytanie użytkownika.
# Wskazówka: połącz komunikat tekstowy z obrazem i przekaż jako wejście do modelu.
from base64 import b64encode
from pathlib import Path
from openai import OpenAI

img_path = Path("data/img/airport.jpg")  # <- podmień na własną ścieżkę
question = "Ile osób znajduje się na zdjęciu?"

if img_path.exists():
    data = b64encode(img_path.read_bytes()).decode("utf-8")
    content = [
        {"type": "input_text", "text": f"Odpowiedz krótko po polsku: ____"}, # <- wstaw question
        {"type": "input_image", "image_url": f"data:image/jpeg;base64,____"} # <- wstaw data
    ]
    client = OpenAI()
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=[{
            "role": "user",
            "content": content
        }])

    print(resp.output_text)
else:
    print("Brak pliku – wstaw własny obraz i uruchom ponownie.")

Nie wiem, ile osób znajduje się na zdjęciu.
